# 🚀 Qwen-Image-2.1 Uncensored (Q4_0 GGUF) 원클릭 검증 생성기

요청하신 **[abenzerps/Qwen-Image-2.1-Uncensored-GGUF](https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF)** 모델 전용 검증 노트북입니다.

---
### 📌 구성 모델 사양
1. **디퓨전 모델**: `qwen-image-2.1-UC-Q4_0.gguf` (4.15 GB)
2. **텍스트 인코더**: `qwen3vl_8b_int8_convrot.safetensors` (9.35 GB, INT8 저메모리 버전)
3. **VAE**: `qwen_image_2.1_vae_bf16.safetensors` (676 MB)
4. **인터페이스**: 복잡한 노드 연결 없이 **프롬프트 입력창 하나로 생성하는 심플 WebUI**

👉 **상단 메뉴에서 `런타임 > 모두 실행(Run all)`**을 클릭하시면 모든 과정이 전자동으로 진행됩니다.

### 1단계: GPU 환경 확인
Colab 상단 메뉴 `런타임 > 런타임 유형 변경`에서 **T4 GPU**로 설정되어 있는지 확인합니다.

In [ ]:
!nvidia-smi

### 2단계: 필수 패키지 및 ComfyUI-GGUF 백엔드 엔진 설치
- Qwen-Image 2.1 네이티브 지원 `leejet/ComfyUI-GGUF` 설치
- 심플 웹 폼을 위한 `gradio` 설치

In [ ]:
# 1. 기존 잔여 GPU 프로세스 정리 (메모리 100% 확보)
!fuser -k -9 /dev/nvidia* > /dev/null 2>&1 || true
!pkill -9 -f "main.py" > /dev/null 2>&1 || true

# 2. 필수 라이브러리 및 Gradio 설치
!apt-get update -qq && apt-get install -y -qq aria2
!pip install -q gradio Pillow gguf huggingface_hub

# 3. ComfyUI 백엔드 클론 및 설치
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || true
%cd /content/ComfyUI
!pip install -q -r requirements.txt

# 4. Qwen-Image-2.1 네이티브 지원 GGUF 노드 설치
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/leejet/ComfyUI-GGUF.git 2>/dev/null || true
%cd /content/ComfyUI/custom_nodes/ComfyUI-GGUF
!pip install -q -r requirements.txt
%cd /content/ComfyUI
print('✅ 환경 설치 완료!')

### 3단계: Qwen-Image-2.1 Uncensored 3종 모델 다운로드
이미 다운로드되어 있다면 1초 만에 건너뜁니다.

In [ ]:
import os

# 폴더 생성
os.makedirs('/content/ComfyUI/models/diffusion_models', exist_ok=True)
os.makedirs('/content/ComfyUI/models/unet', exist_ok=True)
os.makedirs('/content/ComfyUI/models/text_encoders', exist_ok=True)
os.makedirs('/content/ComfyUI/models/clip', exist_ok=True)
os.makedirs('/content/ComfyUI/models/vae', exist_ok=True)

ua = '--user-agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"'

# 1. Qwen-Image-2.1 UC Q4_0 GGUF (4.15 GB)
diff_path = '/content/ComfyUI/models/diffusion_models/qwen-image-2.1-UC-Q4_0.gguf'
if not os.path.exists(diff_path) or os.path.getsize(diff_path) < 1000000:
    print('📥 [1/3] Qwen-Image-2.1 UC Q4_0 GGUF 다운로드 중 (약 4.15 GB)...')
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M $ua \
      'https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF/resolve/main/qwen-image-2.1-UC-Q4_0.gguf' \
      -d /content/ComfyUI/models/diffusion_models -o qwen-image-2.1-UC-Q4_0.gguf
else:
    print('✅ [1/3] Qwen-Image-2.1 UC Q4_0 GGUF 이미 존재함.')

!ln -sf /content/ComfyUI/models/diffusion_models/qwen-image-2.1-UC-Q4_0.gguf /content/ComfyUI/models/unet/ 2>/dev/null || true

# 2. Qwen3-VL 8B INT8 ConvRot Text Encoder (9.35 GB)
clip_path = '/content/ComfyUI/models/text_encoders/qwen3vl_8b_int8_convrot.safetensors'
if not os.path.exists(clip_path) or os.path.getsize(clip_path) < 1000000:
    print('📥 [2/3] Qwen3-VL 8B INT8 Text Encoder 다운로드 중 (약 9.35 GB)...')
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M $ua \
      'https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors' \
      -d /content/ComfyUI/models/text_encoders -o qwen3vl_8b_int8_convrot.safetensors
else:
    print('✅ [2/3] Qwen3-VL 8B INT8 Text Encoder 이미 존재함.')

!ln -sf /content/ComfyUI/models/text_encoders/qwen3vl_8b_int8_convrot.safetensors /content/ComfyUI/models/clip/ 2>/dev/null || true

# 3. Qwen-Image-2.1 VAE (676 MB)
vae_path = '/content/ComfyUI/models/vae/qwen_image_2.1_vae_bf16.safetensors'
if not os.path.exists(vae_path) or os.path.getsize(vae_path) < 1000000:
    print('📥 [3/3] Qwen-Image-2.1 VAE 다운로드 중 (약 676 MB)...')
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M $ua \
      'https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors' \
      -d /content/ComfyUI/models/vae -o qwen_image_2.1_vae_bf16.safetensors
else:
    print('✅ [3/3] Qwen-Image-2.1 VAE 이미 존재함.')

print('🎉 모든 Qwen 2.1 모델 파일 준비 완료!')

### 4단계: Qwen-Image-2.1 전용 심플 웹 생성기 실행
아래 셀을 실행하면 **Colab 화면 바로 아래에 프롬프트 입력창**이 나타납니다!
- **T4 GPU 최적화**: 14GB 대형 모델이므로 첫 실행 시 모델 로드에 약 1~2분이 걸리며, 이후 실시간 진행 시간이 표시됩니다.
- **권장 해상도**: 빠른 검증을 위해 `768x768` (약 1.5~2분) 또는 정밀 검증용 `1024x1024`를 선택할 수 있습니다.

In [ ]:
import subprocess
import time
import urllib.request
import urllib.parse
import urllib.error
import json
import random
import io
import os
import gradio as gr
from PIL import Image

%cd /content/ComfyUI

# 1. 기존 프로세스 종료 및 포트 8188 청소
!fuser -k -9 8188/tcp > /dev/null 2>&1 || true
!pkill -9 -f "main.py" > /dev/null 2>&1 || true
time.sleep(2)

# 2. 백그라운드 ComfyUI 엔진 실행 (--lowvram으로 14GB 모델 안전 구동)
print('⏳ Qwen-Image-2.1 백엔드 엔진 시작 중...')
log_file = open('/content/comfyui.log', 'w')
engine_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188', '--lowvram', '--preview-method', 'none'],
    stdout=log_file,
    stderr=log_file
)

# 엔진 준비 대기 (최대 40초)
ready = False
for i in range(20):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2) as resp:
            if resp.status == 200:
                print('✅ Qwen-Image 백엔드 엔진 준비 완료!')
                ready = True
                break
    except Exception:
        time.sleep(2)

if not ready:
    print('⚠️ 백엔드 시작 로그:')
    if os.path.exists('/content/comfyui.log'):
        with open('/content/comfyui.log') as f:
            print(f.read()[-1000:])

# 3. Qwen-Image-2.1 이미지 생성 함수
def generate_image(prompt, negative_prompt, steps, cfg, width, height, seed, progress=gr.Progress()):
    if not prompt or prompt.strip() == '':
        raise gr.Error('프롬프트를 입력해 주세요!')
    
    if seed == -1 or seed is None:
        seed = random.randint(1, 1000000000)
    
    progress(None, desc='⏳ [1단계] Qwen-Image-2.1 모델 요청 생성 중...')
    
    # Qwen-Image 2.1 네이티브 워크플로우 정의 (type: qwen_image 필수)
    workflow = {
        '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': 'qwen-image-2.1-UC-Q4_0.gguf'}},
        '2': {'class_type': 'CLIPLoader', 'inputs': {'clip_name': 'qwen3vl_8b_int8_convrot.safetensors', 'type': 'qwen_image'}},
        '3': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'qwen_image_2.1_vae_bf16.safetensors'}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'clip': ['2', 0], 'text': prompt}},
        '5': {'class_type': 'CLIPTextEncode', 'inputs': {'clip': ['2', 0], 'text': negative_prompt}},
        '6': {'class_type': 'EmptyLatentImage', 'inputs': {'width': int(width), 'height': int(height), 'batch_size': 1}},
        '7': {'class_type': 'KSampler', 'inputs': {
            'model': ['1', 0],
            'positive': ['4', 0],
            'negative': ['5', 0],
            'latent_image': ['6', 0],
            'seed': int(seed),
            'steps': int(steps),
            'cfg': float(cfg),
            'sampler_name': 'euler',
            'scheduler': 'normal',
            'denoise': 1.0
        }},
        '8': {'class_type': 'VAEDecode', 'inputs': {'samples': ['7', 0], 'vae': ['3', 0]}},
        '9': {'class_type': 'SaveImage', 'inputs': {'images': ['8', 0], 'filename_prefix': 'Qwen2_1_Output'}}
    }

    try:
        data = json.dumps({'prompt': workflow}).encode('utf-8')
        req = urllib.request.Request('http://127.0.0.1:8188/prompt', data=data, headers={'Content-Type': 'application/json'})
        with urllib.request.urlopen(req, timeout=15) as resp:
            resp_data = json.loads(resp.read().decode('utf-8'))
            prompt_id = resp_data['prompt_id']
    except urllib.error.HTTPError as e:
        err_msg = e.read().decode('utf-8')
        raise gr.Error(f'ComfyUI 요청 실패: {err_msg}')
    except Exception as e:
        raise gr.Error(f'요청 전송 실패: {e}')

    # 실시간 진행 시간 표시 (타임아웃 10분)
    filename, subfolder, type_ = None, None, None
    start_time = time.time()
    while time.time() - start_time < 600:
        time.sleep(2)
        elapsed = int(time.time() - start_time)
        if elapsed < 50:
            msg = f'⏳ [1/2] 14GB 대형 모델(9.3GB 텍스트인코더+4.1GB 디퓨전) 로딩 중... ({elapsed}초 경과)'
        else:
            msg = f'🎨 [2/2] Qwen-Image-2.1 연산 중 (약 1.5~3분 소요)... ({elapsed}초 경과)'
        progress(None, desc=msg)
        
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:8188/history/{prompt_id}', timeout=5) as resp:
                history = json.loads(resp.read().decode('utf-8'))
                if prompt_id in history:
                    outputs = history[prompt_id].get('outputs', {})
                    if '9' in outputs and 'images' in outputs['9']:
                        img_info = outputs['9']['images'][0]
                        filename = img_info['filename']
                        subfolder = img_info['subfolder']
                        type_ = img_info['type']
                        break
                    status = history[prompt_id].get('status', {})
                    if status.get('status_str') == 'error':
                        raise gr.Error(f'생성 실패: {status}')
        except gr.Error:
            raise
        except Exception:
            pass

    if not filename:
        raise gr.Error('이미지 생성 시간 초과 또는 오류가 발생했습니다.')

    progress(None, desc='✨ 이미지 생성 완료! 화면에 표시하는 중...')
    params = urllib.parse.urlencode({'filename': filename, 'subfolder': subfolder, 'type': type_})
    with urllib.request.urlopen(f'http://127.0.0.1:8188/view?{params}', timeout=15) as resp:
        return Image.open(io.BytesIO(resp.read()))

# 4. Gradio 웹 UI 생성
with gr.Blocks(theme=gr.themes.Soft(), title='Qwen-Image-2.1 Uncensored 검증기') as demo:
    gr.Markdown('# 🚀 Qwen-Image-2.1 Uncensored (Q4_0 GGUF) 모델 검증기')
    gr.Markdown('💡 **안내**: Qwen 2.1은 9.3GB 텍스트 인코더와 4.1GB 디퓨전 모델로 구성되어 첫 1회 실행 시 메모리 로드에 1~2분이 걸립니다. 편안하게 기다려 주세요!')
    
    with gr.Row():
        with gr.Column(scale=1):
            prompt_box = gr.Textbox(
                label='📝 프롬프트 (그릴 내용)',
                placeholder='영어로 프롬프트 입력',
                lines=4,
                value='a beautiful anime girl with long silver hair in a cherry blossom garden, sunny day, highly detailed, 8k masterpiece'
            )
            neg_prompt_box = gr.Textbox(
                label='🚫 부정 프롬프트 (제외할 내용)',
                lines=2,
                value='low quality, blurry, distorted, deformed, bad anatomy, worst quality'
            )
            
            with gr.Accordion('⚙️ 상세 옵션 (해상도 및 스텝 조절)', open=True):
                with gr.Row():
                    width_slider = gr.Slider(512, 1024, value=768, step=64, label='가로 해상도 (빠른 검증: 768 / 정밀: 1024)')
                    height_slider = gr.Slider(512, 1024, value=768, step=64, label='세로 해상도 (빠른 검증: 768 / 정밀: 1024)')
                steps_slider = gr.Slider(10, 30, value=20, step=1, label='생성 스텝수 (권장 20)')
                cfg_slider = gr.Slider(1.0, 10.0, value=4.0, step=0.5, label='CFG (권장 3.5~4.5)')
                seed_input = gr.Number(value=-1, label='시드 (-1은 랜덤)')
            
            generate_btn = gr.Button('🚀 Qwen-Image 2.1 이미지 생성하기', variant='primary', size='lg')
            
        with gr.Column(scale=1):
            output_img = gr.Image(label='🖼️ 생성된 Qwen-Image 결과물', type='pil', interactive=False)
            
    generate_btn.click(
        fn=generate_image,
        inputs=[prompt_box, neg_prompt_box, steps_slider, cfg_slider, width_slider, height_slider, seed_input],
        outputs=output_img
    )

demo.queue().launch(share=True, debug=False)
